# Stage 5: Multi-resolution Leiden clustering + sweep

Clustering is run at multiple Leiden resolutions and compared so PI
can choose the resolution that best captures the biological grain.

**Default**: multi-resolution Leiden on the promoted stage-4 embedding.

**Extension slot** (commented cell): any clustering method writing
`obs["{method}_clusters"]` coexists with `leiden_res_*` columns.
ACDC is NOT a default dependency but plugs in as one added cell.

**What this notebook produces**:
- `obs["leiden_res_{resolution}"]` columns
- UMAP plots coloured by each resolution
- Sweep report with `clustering_metrics`
- Stage 5 checkpoint h5ad ready for stage 6

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH  -- stage 4 (embedded) output.
# OUTPUT_PATH    -- where to write this stage's checkpoint.
# USE_REP        -- which obsm embedding for the neighbor graph.
# RESOLUTIONS    -- Leiden resolutions to try.
# RANDOM_SEED    -- fixed for reproducibility.

UPSTREAM_PATH = "results/nancang_stage4_embedded_v1.h5ad"
OUTPUT_PATH   = "results/nancang_stage5_clustered_v1.h5ad"

USE_REP      = "X_scVI"      # embedding for neighbor graph
RESOLUTIONS  = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]
RANDOM_SEED  = 42

In [ ]:
# Ensure the framework src/ is on sys.path and CWD is set to the project root.
import sys, os
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/figures/sweep_stage5", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

In [ ]:
# Imports (scanpy native API + framework functions).
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import matplotlib.pyplot as plt
import datetime
import warnings

from scrna_integration import sweep
from scrna_integration.scorers import clustering_metrics

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

print("Loading upstream:", UPSTREAM_PATH)
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obsm keys: {list(adata.obsm.keys())}")
print(f"use_rep='{USE_REP}' -- present: {USE_REP in adata.obsm}")

## Compute neighbor graph

Build the k-nearest-neighbor graph on the stage-4 embedding chosen in
`USE_REP`. This graph is shared across all Leiden resolutions.

In [ ]:
# Compute neighbors on the promoted embedding.
# n_pcs=None uses all dimensions of the embedding (already low-dimensional).
print(f"\n===== Neighbors on {USE_REP} =====")
sc.pp.neighbors(
    adata,
    use_rep=USE_REP,
    n_pcs=None,
    random_state=RANDOM_SEED,
)
print(f"Neighbor graph: {adata.obsp['connectivities'].shape}")
print(f"  n_neighbors={adata.uns['neighbors']['params']['n_neighbors']}")

## Multi-resolution Leiden clustering

Leiden clustering at each resolution. All columns coexist for comparison.
`flavor="igraph"` for forward compatibility with scanpy >= 1.10.

In [ ]:
# Multi-resolution Leiden: one column per resolution.
print(f"\n===== Leiden: {len(RESOLUTIONS)} resolutions =====")

for res in RESOLUTIONS:
    key = f"leiden_res_{res}"
    print(f"  resolution={res} -> obs['{key}']")
    sc.tl.leiden(
        adata,
        resolution=res,
        key_added=key,
        flavor="igraph",
        random_state=RANDOM_SEED,
    )
    n_clusters = adata.obs[key].nunique()
    print(f"    clusters: {n_clusters}")

leiden_columns = [c for c in adata.obs.columns if c.startswith("leiden_res_")]
print(f"\nLeiden columns produced: {leiden_columns}")

## Sweep across resolutions with clustering_metrics

Quantitative clustering quality metrics via `sweep()`.

Metrics (when data permits):
- **silhouette** -- cluster separation on PCA
- **ari** -- Adjusted Rand Index against known cell-type labels

Report written to `results/figures/sweep_stage5/sweep_report.md`.

In [ ]:
# Sweep: run Leiden at each resolution and score.
print("\n===== Sweep: clustering_metrics =====")

def leiden_wrapper(_adata, resolution):
    key = f"leiden_res_{resolution}"
    sc.tl.leiden(
        _adata,
        resolution=resolution,
        key_added=key,
        flavor="igraph",
        random_state=RANDOM_SEED,
    )

sweep_df = sweep(
    fn=leiden_wrapper,
    adata=adata,
    candidates={"resolution": RESOLUTIONS},
    scorer=clustering_metrics,
    output_dir="results/figures/sweep_stage5",
)

print("\nClustering metrics table:")
try:
    from IPython.display import display as ipy_display
    ipy_display(sweep_df)
except ImportError:
    print(sweep_df)

print("\nSweep report: results/figures/sweep_stage5/sweep_report.md")
adata.uns["stage5_sweep_v1"] = {
    "resolutions_swept": RESOLUTIONS,
    "scorer": "clustering_metrics",
    "report_dir": "results/figures/sweep_stage5",
    "timestamp": datetime.datetime.now().isoformat(),
}

## (Optional) Alternative clustering method -- extension slot

Any clustering method that writes `obs["{method}_clusters"]` coexists
with `leiden_res_*` columns and is compared the same way.

**ACDC** (commented out): searches for globally optimal partition.
NOT a default dependency (too slow on GCPL). When PI installs it,
uncomment the cell and the new column flows into stage 6 automatically.

**Pattern for any new method**: write labels to `obs["{method}_clusters"]`,
optionally add to sweep candidates -- same parallel-slots convention
as stage 4 embeddings, zero framework change.

In [ ]:
# # === ACDC clustering (commented out -- NOT a default dependency) ===
# # PREREQUISITE: pip install acdc_py
# # Enable by removing comments below.
#
# # import acdc_py  # 包名/导入名以 PyPI 实际为准，启用前先确认；ACDC 非默认依赖
# # # ACDC searches for an optimal partition.
# # acdc_result = ACDC.ACDC(adata, ...)
# # adata.obs["acdc_clusters"] = acdc_result.labels
# # print(f"ACDC: {adata.obs['acdc_clusters'].nunique()} clusters")
#
# print("ACDC cell is commented out. "
#       "Uncomment when acdc_py is installed and suitable for this dataset.")
#
# # === Add any future method here ===
# # Pattern: write adata.obs["{method}_clusters"] = labels
# # Then add to sweep candidates or compare standalone.
# # Example: adata.obs["foocluster_clusters"] = foo_cluster.fit_predict(
# #     adata.obsm[USE_REP])

## Visualisation -- UMAP coloured by each Leiden resolution

UMAP plots coloured by cluster assignment at each resolution.
PI compares to choose the resolution that best captures biological structure.

In [ ]:
# UMAP coloured by each leiden_res_* column.
print("\n===== UMAP per resolution =====")

leiden_cols = sorted(
    [c for c in adata.obs.columns if c.startswith("leiden_res_")],
    key=lambda x: float(x.split("_")[-1]),
)

# Compute UMAP from existing neighbor graph.
sc.tl.umap(adata, random_state=RANDOM_SEED)

for col in leiden_cols:
    n_clusters = adata.obs[col].nunique()
    print(f"  {col}: {n_clusters} clusters")

    sc.pl.umap(
        adata, color=col,
        title=f"Leiden res={col.split('_')[-1]} ({n_clusters} clusters)",
        legend_loc="on data" if n_clusters <= 10 else "right margin",
        frameon=False,
        save=f"_stage5_{col}.png",
    )
    # Move from scanpy default figures/ dir to results/figures/.
    src = f"figures/umap_stage5_{col}.png"
    dst = f"results/figures/stage5_umap_{col}.png"
    if os.path.exists(src):
        os.rename(src, dst)
        print(f"    Saved {dst}")
    plt.close("all")

print(f"\nUMAP plots saved for {len(leiden_cols)} resolutions.")

## Run metadata -- plain `adata.uns` writes

Versioned keys record clustering parameters for traceability.

In [ ]:
# Record clustering run metadata.
print("\n===== Run metadata =====")

adata.uns["leiden_v1"] = {
    "use_rep": USE_REP,
    "resolutions": RESOLUTIONS,
    "flavor": "igraph",
    "n_neighbors": adata.uns["neighbors"]["params"]["n_neighbors"],
    "timestamp": datetime.datetime.now().isoformat(),
}

adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"

# Cluster count summary.
cluster_summary = {}
for col in leiden_cols:
    cluster_summary[col] = int(adata.obs[col].nunique())
adata.uns["leiden_v1"]["cluster_counts"] = cluster_summary

print("Clusters per resolution:")
for col, n in cluster_summary.items():
    print(f"  {col}: {n}")
print(f"status: {adata.uns['status']}")

In [ ]:
# Memory discipline self-check (one assertion before write).
# Guards the highest-impact memory regression.
import scipy.sparse as sp
import numpy as np
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X invariants violated: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("Memory self-check passed: X is sparse CSR float32.")

In [ ]:
# Write the stage checkpoint to disk (Memory Discipline #4: lzf compression).
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"Wrote {OUTPUT_PATH}")

import os
assert os.path.exists(OUTPUT_PATH), f"Output NOT found: {OUTPUT_PATH}"
print(f"Verified: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# Free memory across stage boundaries (Memory Discipline #3).
del adata
import gc
gc.collect()
print("Memory released.")